# Scoring de la campagne 500 scenarios x 5 modeles x 3 strategies

Ce notebook calcule deux choses, separement, pour ne pas les confondre :

1. **Exactitude de l'extraction** -- ce que le LLM a produit (`extraction_raw`,
   avant validation/normalisation) et ce que le pipeline a reellement garde
   (`extracted_requirements`, apres `validate_and_normalize`), compares tous
   les deux a la verite terrain du scenario (`ground_truth.requirements`,
   deja en unite canonique -- vCPU/GB/ms).
2. **Exactitude de la decision finale** -- la classification STAY/OFFLOAD/
   INFEASIBLE produite par necessity_checker + decision_engine, comparee a
   `ground_truth.decision`, et (quand la verite terrain dit OFFLOAD) la
   liste exacte des services deplaces.

Les 8 scenarios qui echouent encore apres la correction de `max_tokens`
(tous `gpt-oss-120b`, boucle de generation incontrolee meme a 8192 tokens)
sont exclus du scoring -- ils n'ont pas de resultat exploitable, ce n'est
pas une erreur d'extraction ou de decision.


In [ ]:
import json
import math
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent  # repo root, assuming this notebook runs from eval/
sys.path.insert(0, str(ROOT / "src"))

from agent.utils.pipeline import to_native_unit  # noqa: E402

RESULTS_DIR = ROOT / "eval" / "results"

# "e2e_latency" (ground_truth.requirements' own kpi_type spelling) ->
# "latency" (our schema's spelling) -- same remap intent_grounding_node's
# own ground truth comparisons would need, see loader.py.
KPI_REMAP = {"e2e_latency": "latency"}

# A value counts as a match if within 1% of the ground truth (converted to
# the same canonical unit) -- loose enough to absorb float rounding from
# unit conversion, tight enough that a genuinely wrong digit still misses.
VALUE_REL_TOL = 0.01

# The 15 campaign files -- excludes the leftover 5-record smoke-test file
# from before the full launch (mistralai...__zero_shot__20261103T214637Z).
RESULT_FILES = [
    "google-gemini-2.5-flash-lite__zero_shot__20261103T220001Z.jsonl",
    "google-gemini-2.5-flash-lite__one_shot__20261103T220001Z.jsonl",
    "google-gemini-2.5-flash-lite__few_shot__20261103T220001Z.jsonl",
    "mistralai-mistral-small-3.2-24b-instruct__zero_shot__20261103T220001Z.jsonl",
    "mistralai-mistral-small-3.2-24b-instruct__one_shot__20261103T220001Z.jsonl",
    "mistralai-mistral-small-3.2-24b-instruct__few_shot__20261103T220001Z.jsonl",
    "openai-gpt-oss-20b__zero_shot__20261103T220000Z.jsonl",
    "openai-gpt-oss-20b__one_shot__20261103T220001Z.jsonl",
    "openai-gpt-oss-20b__few_shot__20261103T220001Z.jsonl",
    "meta-llama-llama-3.3-70b-instruct__zero_shot__20261103T220001Z.jsonl",
    "meta-llama-llama-3.3-70b-instruct__one_shot__20261103T220001Z.jsonl",
    "meta-llama-llama-3.3-70b-instruct__few_shot__20261103T220001Z.jsonl",
    "openai-gpt-oss-120b__zero_shot__20261103T220001Z.jsonl",
    "openai-gpt-oss-120b__one_shot__20261103T220001Z.jsonl",
    "openai-gpt-oss-120b__few_shot__20261103T220001Z.jsonl",
]

MODEL_SHORT_NAME = {
    "google/gemini-2.5-flash-lite": "gemini-2.5-flash-lite",
    "mistralai/mistral-small-3.2-24b-instruct": "mistral-small-3.2-24b",
    "openai/gpt-oss-20b": "gpt-oss-20b",
    "meta-llama/llama-3.3-70b-instruct": "llama-3.3-70b",
    "openai/gpt-oss-120b": "gpt-oss-120b",
}


## Chargement -- un enregistrement par scenario reussi (erreurs exclues)

In [ ]:
records = []
n_errors_excluded = 0
for name in RESULT_FILES:
    with open(RESULTS_DIR / name) as f:
        for line in f:
            r = json.loads(line)
            if "error" in r:
                n_errors_excluded += 1
                continue
            records.append(r)

print(f"{len(records)} scenarios reussis charges, {n_errors_excluded} erreurs exclues "
      f"(sur {len(records) + n_errors_excluded} scenarios attendus)")


## Fonctions de scoring

**Extraction** : chaque exigence de la verite terrain (`ground_truth.requirements`,
deja en unite canonique) est appariee a une exigence predite par
`(target_id, kpi_type, comparator)` identiques et une valeur a moins de 1%
pres. Calcule separement pour `extraction_raw` (il faut alors convertir
nous-memes vers l'unite canonique, avec `to_native_unit` -- la meme fonction
que le pipeline reel utilise) et pour `extracted_requirements` (deja
converti par `validate_and_normalize`, comparable directement).

**Decision** : necessity_checker/decision_engine ne renvoient pas directement
un label STAY/OFFLOAD/INFEASIBLE -- on le reconstruit depuis `necessity_result`
et `decision_result`, avec la meme regle que `explanation.py:build_user_prompt`
applique deja pour decider quoi dire a l'operateur (`decision_result is None`
ou `decision=={}` => rien a faire => STAY).


In [ ]:
def gt_requirements(record):
    out = []
    for req in record["ground_truth"]["requirements"]:
        kpi = KPI_REMAP.get(req["kpi_type"], req["kpi_type"])
        out.append({"target_id": req["target_id"], "kpi_type": kpi,
                     "comparator": req["comparator"], "value": req["new_value"]})
    return out


def match_requirements(gt_list, pred_list):
    """pred_list items must already carry a canonical `value`. Returns (tp, fp, fn)."""
    matched_gt = set()
    tp = 0
    for p in pred_list:
        for i, g in enumerate(gt_list):
            if i in matched_gt:
                continue
            if (g["target_id"] == p["target_id"] and g["kpi_type"] == p["kpi_type"]
                    and g["comparator"] == p["comparator"]
                    and math.isclose(g["value"], p["value"], rel_tol=VALUE_REL_TOL, abs_tol=1e-6)):
                matched_gt.add(i)
                tp += 1
                break
    fp = len(pred_list) - tp
    fn = len(gt_list) - len(matched_gt)
    return tp, fp, fn


def extracted_to_canonical(req):
    return {"target_id": req["target_id"], "kpi_type": req["kpi_type"],
            "comparator": req["comparator"], "value": req["target_value"]}


def raw_to_canonical(req):
    native = to_native_unit(req["target_value"], req["unit"], req["kpi_type"])
    if native is None:
        return None
    return {"target_id": req["target_id"], "kpi_type": req["kpi_type"],
            "comparator": req["comparator"], "value": native}


def predicted_decision(record):
    necessity = record["necessity_result"]
    decision = record["decision_result"]
    if necessity is not None and necessity["status"] == "reconfiguration_not_required":
        return "STAY"
    if decision is None:
        return "STAY"
    if decision["outcome"] == "INFEASIBLE":
        return "INFEASIBLE"
    if not decision["decision"]:
        return "STAY"
    return "OFFLOAD"


def predicted_moved(record):
    decision = record["decision_result"]
    if decision is None or not decision.get("decision"):
        return frozenset()
    return frozenset(decision["decision"].keys())


def gt_moved(record):
    return frozenset(f"C{n}" for n in record["ground_truth"]["moved_components"])


## Construction du tableau par scenario

In [ ]:
rows = []
for r in records:
    gt = gt_requirements(r)

    pred_ext = [extracted_to_canonical(x) for x in r["extracted_requirements"]]
    tp_e, fp_e, fn_e = match_requirements(gt, pred_ext)

    pred_raw = [raw_to_canonical(x) for x in r["extraction_raw"]]
    pred_raw = [x for x in pred_raw if x is not None]
    tp_r, fp_r, fn_r = match_requirements(gt, pred_raw)

    pdec = predicted_decision(r)
    gdec = r["ground_truth"]["decision"]
    pmoved = predicted_moved(r)
    gmoved = gt_moved(r)

    rows.append({
        "model": r["run_config"]["model"],
        "model_short": MODEL_SHORT_NAME[r["run_config"]["model"]],
        "strategy": r["run_config"]["prompt_strategy"],
        "scenario_id": r["scenario_id"],
        "n_gt_requirements": len(gt),
        "tp_ext": tp_e, "fp_ext": fp_e, "fn_ext": fn_e,
        "tp_raw": tp_r, "fp_raw": fp_r, "fn_raw": fn_r,
        "gt_decision": gdec,
        "pred_decision": pdec,
        "decision_correct": pdec == gdec,
        "gt_moved": gmoved,
        "pred_moved": pmoved,
        "moved_correct": pmoved == gmoved if gdec == "OFFLOAD" else None,
        "exact_match_ext": fp_e == 0 and fn_e == 0,
        "exact_match_raw": fp_r == 0 and fn_r == 0,
        "total_elapsed_s": r.get("total_elapsed_s"),
        "extraction_elapsed_s": r.get("extraction_elapsed_s"),
    })

df = pd.DataFrame(rows)
df.shape


## Extraction -- precision / recall / F1 par modele x strategie

In [ ]:
def prf(tp, fp, fn):
    p = tp / (tp + fp) if (tp + fp) else float("nan")
    r = tp / (tp + fn) if (tp + fn) else float("nan")
    f1 = 2 * p * r / (p + r) if (p + r) else float("nan")
    return pd.Series({"precision": p, "recall": r, "f1": f1})


def extraction_summary(df, prefix):
    g = df.groupby(["model_short", "strategy"])[[f"tp_{prefix}", f"fp_{prefix}", f"fn_{prefix}"]].sum()
    return g.apply(lambda row: prf(row[f"tp_{prefix}"], row[f"fp_{prefix}"], row[f"fn_{prefix}"]), axis=1)


print("=== extraction_raw (avant validation) vs ground truth ===")
raw_summary = extraction_summary(df, "raw")
display(raw_summary.round(3))


In [ ]:
print("=== extracted_requirements (apres validate_and_normalize) vs ground truth ===")
ext_summary = extraction_summary(df, "ext")
display(ext_summary.round(3))


**Lecture** : `extraction_raw` mesure ce que le LLM sait vraiment extraire ;
`extracted_requirements` mesure ce que le pipeline reel utilise en pratique
(apres que `validate_and_normalize` ait rejete les cas invalides -- cela peut
AUGMENTER la precision en jetant les hallucinations, mais aussi baisser le
recall si une extraction correcte est rejetee par une regle trop stricte).


In [ ]:
delta = (ext_summary["f1"] - raw_summary["f1"]).rename("f1_delta (ext - raw)")
display(delta.round(3).sort_values())


## Extraction -- exact match par scenario (tout ou rien)

Le F1 ci-dessus est une moyenne par exigence -- un scenario a 4 exigences ou
3 sont bonnes et 1 manque contribue partiellement. L'exact match est plus
strict : un scenario ne compte comme correct que si **toutes** ses exigences
de verite terrain sont retrouvees **et** qu'aucune exigence en trop n'est
extraite (`fp==0 ET fn==0`) -- une exigence hallucinee fait echouer le
scenario entier, meme si le reste est parfait.

In [ ]:
exact_match_summary = df.groupby(["model_short", "strategy"])[["exact_match_raw", "exact_match_ext"]].mean()
display(exact_match_summary.round(3))

### Exact match selon le nombre d'exigences par intent

Teste si la difficulte grandit avec le nombre d'exigences a extraire
simultanement -- global (tous modeles/strategies confondus), puis detaille
par modele.

In [ ]:
exact_match_by_nreq = df.groupby("n_gt_requirements")[["exact_match_raw", "exact_match_ext"]].mean()
exact_match_by_nreq["n_scenarios"] = df.groupby("n_gt_requirements").size()
display(exact_match_by_nreq.round(3))

In [ ]:
exact_match_by_nreq_model = df.groupby(["model_short", "n_gt_requirements"])["exact_match_ext"].mean().unstack("n_gt_requirements")
display(exact_match_by_nreq_model.round(3))

## Extraction -- verite terrain alternative : `mentions` (texte ecrit, unite non canonique)

Jusqu'ici la verite terrain etait `ground_truth.requirements` -- deja en
unite canonique (vCPU/GB/ms), la representation interne du scenario. Ici on
utilise `intent_metadata.mentions` a la place : `written_value`/`written_unit`,
c'est-a-dire litteralement ce qui est ecrit dans le texte de l'intent, avant
toute conversion. Ca demande de convertir nous-memes avec `to_native_unit`
(verifie plus tot : 1057/1057 mentions convertissent sans ecart avec
`ground_truth.requirements`, une fois `pipeline.py` corrige pour
`virtual CPUs`/`milliseconds`/`microseconds`).

L'interet : chaque mention porte aussi `level` (1-4, explicitation) et
`designation` (id/name/role/flow/context) -- on peut donc croiser
l'exactitude de l'extraction avec la difficulte de la formulation, ce que
`ground_truth.requirements` ne permet pas (il n'a pas cette info).

In [ ]:
def gt_requirements_from_mentions(record):
    out = []
    for m in record["intent_metadata"]["mentions"]:
        kpi = KPI_REMAP.get(m["kpi_type"], m["kpi_type"])
        native = to_native_unit(float(m["written_value"]), m["written_unit"], kpi)
        out.append({"target_id": m["target_id"], "kpi_type": kpi, "comparator": m["comparator"],
                     "value": native, "level": m["level"], "designation": m["designation"]})
    return out


def match_requirements_detailed(gt_list, pred_list):
    """Same matching rule as match_requirements, but also returns which
    gt_list entries got matched (by index) -- needed to break recall down
    by a gt-side attribute (level, designation) rather than just summing
    tp/fp/fn."""
    matched_gt = set()
    tp = 0
    for p in pred_list:
        for i, g in enumerate(gt_list):
            if i in matched_gt:
                continue
            if (g["target_id"] == p["target_id"] and g["kpi_type"] == p["kpi_type"]
                    and g["comparator"] == p["comparator"]
                    and math.isclose(g["value"], p["value"], rel_tol=VALUE_REL_TOL, abs_tol=1e-6)):
                matched_gt.add(i)
                tp += 1
                break
    fp = len(pred_list) - tp
    fn = len(gt_list) - len(matched_gt)
    return tp, fp, fn, matched_gt


In [ ]:
mention_rows = []
ext_m_cols = {"tp": [], "fp": [], "fn": []}
raw_m_cols = {"tp": [], "fp": [], "fn": []}

for r in records:
    gt_m = gt_requirements_from_mentions(r)

    pred_ext = [extracted_to_canonical(x) for x in r["extracted_requirements"]]
    tp_e, fp_e, fn_e, matched_e = match_requirements_detailed(gt_m, pred_ext)
    ext_m_cols["tp"].append(tp_e); ext_m_cols["fp"].append(fp_e); ext_m_cols["fn"].append(fn_e)

    pred_raw = [x for x in (raw_to_canonical(x) for x in r["extraction_raw"]) if x is not None]
    tp_r, fp_r, fn_r, matched_r = match_requirements_detailed(gt_m, pred_raw)
    raw_m_cols["tp"].append(tp_r); raw_m_cols["fp"].append(fp_r); raw_m_cols["fn"].append(fn_r)

    for i, g in enumerate(gt_m):
        mention_rows.append({
            "model_short": MODEL_SHORT_NAME[r["run_config"]["model"]],
            "strategy": r["run_config"]["prompt_strategy"],
            "scenario_id": r["scenario_id"],
            "level": g["level"],
            "designation": g["designation"],
            "matched_ext": i in matched_e,
            "matched_raw": i in matched_r,
        })

df["tp_ext_m"] = ext_m_cols["tp"]; df["fp_ext_m"] = ext_m_cols["fp"]; df["fn_ext_m"] = ext_m_cols["fn"]
df["tp_raw_m"] = raw_m_cols["tp"]; df["fp_raw_m"] = raw_m_cols["fp"]; df["fn_raw_m"] = raw_m_cols["fn"]
df["exact_match_ext_m"] = (df["fp_ext_m"] == 0) & (df["fn_ext_m"] == 0)
df["exact_match_raw_m"] = (df["fp_raw_m"] == 0) & (df["fn_raw_m"] == 0)

mentions_df = pd.DataFrame(mention_rows)
mentions_df.shape


### Verification -- F1 et exact match sont-ils proches de ceux obtenus avec `ground_truth.requirements` ?

Doivent etre tres proches (les deux verites terrain representent la meme
valeur reelle) -- un ecart important signalerait un bug de conversion,
pas une vraie difference de difficulte.

In [ ]:
mentions_summary = pd.DataFrame({
    "f1_raw_mentions": extraction_summary(df, "raw_m")["f1"],
    "f1_ext_mentions": extraction_summary(df, "ext_m")["f1"],
    "exact_match_raw_mentions": df.groupby(["model_short", "strategy"])["exact_match_raw_m"].mean(),
    "exact_match_ext_mentions": df.groupby(["model_short", "strategy"])["exact_match_ext_m"].mean(),
})
display(mentions_summary.round(3))


### Rappel de l'extraction selon le niveau d'explicitation de la mention

Ici la granularite est la MENTION individuelle (pas le scenario) -- une
mention de niveau 4 est-elle vraiment plus dure a extraire qu'une de
niveau 1 ? `mentions_df` a une ligne par (scenario, modele, strategie,
mention).

In [ ]:
recall_by_level = mentions_df.groupby("level")[["matched_raw", "matched_ext"]].mean()
recall_by_level["n_mentions"] = mentions_df.groupby("level").size()
display(recall_by_level.round(3))


In [ ]:
recall_by_level_model = mentions_df.groupby(["model_short", "level"])["matched_ext"].mean().unstack("level")
display(recall_by_level_model.round(3))


### Rappel de l'extraction selon le mode de designation de la cible

In [ ]:
recall_by_designation = mentions_df.groupby("designation")[["matched_raw", "matched_ext"]].mean()
recall_by_designation["n_mentions"] = mentions_df.groupby("designation").size()
display(recall_by_designation.round(3))


### Exact match (verite terrain = mentions) selon le niveau d'explicitation MAX du scenario

Meme reduction scenario -> niveau que pour la table "exact match selon le
nombre d'exigences" plus haut (niveau max des mentions du scenario) --
mais cette fois avec l'exact match (0 manquant, 0 en trop), pas seulement
le rappel par mention.

In [ ]:
scenario_max_level = mentions_df.groupby(["model_short", "strategy", "scenario_id"])["level"].max()
df_with_level = df.set_index(["model_short", "strategy", "scenario_id"]).join(
    scenario_max_level.rename("scenario_max_level")
).reset_index()

exact_match_by_level_model = df_with_level.groupby(["model_short", "scenario_max_level"])["exact_match_ext_m"].mean().unstack("scenario_max_level")
display(exact_match_by_level_model.round(3))


## Decision STAY/OFFLOAD/INFEASIBLE -- exactitude par modele x strategie

In [ ]:
decision_acc = df.groupby(["model_short", "strategy"])["decision_correct"].mean()
display(decision_acc.round(3).unstack("strategy"))


## Composants deplaces -- exactitude (seulement quand ground truth = OFFLOAD)

In [ ]:
moved_acc = df[df["gt_decision"] == "OFFLOAD"].groupby(["model_short", "strategy"])["moved_correct"].mean()
display(moved_acc.round(3).unstack("strategy"))


## Matrices de confusion de la decision, par modele (agregees sur les 3 strategies)

In [ ]:
for model in df["model_short"].unique():
    sub = df[df["model_short"] == model]
    cm = pd.crosstab(sub["gt_decision"], sub["pred_decision"], rownames=["ground truth"], colnames=["predit"])
    print(f"--- {model} ---")
    display(cm)


## Optimalite de la decision -- respect REEL des contraintes, et nombre de migrations

Jusqu'ici, "decision correcte" voulait dire "le label STAY/OFFLOAD/INFEASIBLE
predit correspond a `ground_truth.decision`". Mais un label correct ne
garantit pas que la configuration finale choisie satisfait reellement les
VRAIES contraintes du scenario -- le pipeline raisonne sur les exigences
*extraites* (potentiellement legerement fausses), pas sur
`ground_truth.requirements`.

Ici on reverifie chaque configuration finale (`decision_result.new_configuration`,
ou le placement courant si necessity_checker a conclu qu'aucun changement
n'etait necessaire) avec `check_candidate_placement` -- la MEME fonction
que necessity_checker/decision_engine utilisent en production, mais appliquee
aux VRAIES exigences du scenario (`ground_truth.requirements`, deja
canoniques), pas a celles extraites par le LLM.

**Important : INFEASIBLE est exclu de `real_valid`.** Par construction, si
la verite terrain dit INFEASIBLE, AUCUNE configuration (meme la bonne
reponse) ne peut satisfaire `check_candidate_placement` -- c'est la
definition meme d'INFEASIBLE. Un `real_valid=False` sur un scenario
INFEASIBLE ne veut donc rien dire de mal ; seuls STAY et OFFLOAD ont un
sens ici.

Verifie au prealable (independamment de tout modele) que la methode est
coherente avec le benchmark lui-meme : sur les 500 scenarios, le vrai
placement de reference (p0 pour STAY/INFEASIBLE, p1 pour OFFLOAD) ne
diverge de `check_candidate_placement` que sur 13/500 cas (2.6%), tous dus
a l'arrondi `floor()` des capacites de noeuds (choix documente dans
`loader.py`, pas une erreur de methode).

In [ ]:
sys.path.insert(0, str(ROOT / "eval" / "system_model"))
from agent.model.schemas import Requirement  # noqa: E402
from agent.utils.pipeline import check_candidate_placement  # noqa: E402
from loader import load_flows, load_links, load_nodes, load_services  # noqa: E402

with open(ROOT / "eval" / "experiment_500" / "scenarios.json") as f:
    SCENARIOS_BY_ID = {s["id"]: s for s in json.load(f)}

NODES = load_nodes()
LINKS = load_links()
_fixtures_cache = {}


def scenario_fixtures(scenario_id):
    """services/flows/ground-truth-Requirements for one scenario, cached --
    500 unique scenarios shared across the 15 result files, not recomputed
    per record."""
    if scenario_id not in _fixtures_cache:
        scenario = SCENARIOS_BY_ID[scenario_id]
        services = load_services(scenario)
        flows = load_flows(scenario)
        gt_reqs = []
        for i, req in enumerate(scenario["requirements"]):
            kpi = KPI_REMAP.get(req["kpi_type"], req["kpi_type"])
            gt_reqs.append(Requirement(
                kpi_type=kpi, comparator=req["comparator"], target_value=req["new_value"],
                unit=req["unit"], target_type=req["target_type"], target_id=req["target_id"],
                source_span="", requirement_id=f"gt-{i}",
            ))
        _fixtures_cache[scenario_id] = (services, flows, gt_reqs)
    return _fixtures_cache[scenario_id]


def final_configuration(record):
    if record["decision_result"] is not None:
        return record["decision_result"]["new_configuration"]
    return record["necessity_result"]["current_placement"]


In [ ]:
real_valid = []
migration_excess = []  # None unless gt_decision == OFFLOAD and real_valid

for r in records:
    services, flows, gt_reqs = scenario_fixtures(r["scenario_id"])
    cfg = final_configuration(r)
    valid, _checks = check_candidate_placement(cfg, services, NODES, LINKS, gt_reqs, flows)
    real_valid.append(valid)

    dr = r["decision_result"]
    if (r["ground_truth"]["decision"] == "OFFLOAD" and valid
            and dr is not None and dr["outcome"] == "FEASIBLE" and dr["decision"]):
        migration_excess.append(len(dr["decision"]) - r["ground_truth"]["minimum_migrations"])
    else:
        migration_excess.append(None)

df["real_valid"] = real_valid
df["migration_excess"] = migration_excess
df[["gt_decision", "real_valid"]].groupby("gt_decision").mean(numeric_only=True)


### Taux de validite reelle, par modele x strategie x decision attendue

`INFEASIBLE` est affiche pour verification (doit rester a 0.000 partout --
sinon la methode a un probleme), mais seules les colonnes `STAY`/`OFFLOAD`
sont a lire comme une mesure de qualite.

In [ ]:
real_valid_by_decision = df.groupby(["model_short", "strategy", "gt_decision"])["real_valid"].mean().unstack("gt_decision")
display(real_valid_by_decision.round(3)[["STAY", "OFFLOAD", "INFEASIBLE"]])


### Optimalite du nombre de migrations (seulement OFFLOAD + configuration reellement valide)

`migration_excess = 0` veut dire que le nombre de services deplaces est
exactement le minimum necessaire (`ground_truth.minimum_migrations`) --
donc que CP-SAT a trouve une solution optimale, pas seulement une solution
qui marche.

In [ ]:
valid_offload = df[(df["gt_decision"] == "OFFLOAD") & df["real_valid"]].copy()
valid_offload["exact_optimal"] = valid_offload["migration_excess"] == 0

migration_summary = valid_offload.groupby(["model_short", "strategy"]).agg(
    n=("migration_excess", "size"),
    mean_excess=("migration_excess", "mean"),
    exact_optimal_rate=("exact_optimal", "mean"),
)
display(migration_summary.round(3))


## Vue d'ensemble -- tableau recapitulatif par modele x strategie

In [ ]:
summary = pd.DataFrame({
    "f1_raw": raw_summary["f1"],
    "f1_extracted": ext_summary["f1"],
    "decision_accuracy": decision_acc,
    "moved_accuracy_when_offload": moved_acc,
    "exact_match_ext": exact_match_summary["exact_match_ext"],
    "exact_match_raw": exact_match_summary["exact_match_raw"],
    "real_valid_offload": real_valid_by_decision["OFFLOAD"],
    "migration_exact_optimal": migration_summary["exact_optimal_rate"],
    "exact_match_ext_mentions": df.groupby(["model_short", "strategy"])["exact_match_ext_m"].mean(),
})
display(summary.round(3))


## Graphiques -- F1 extraction et exactitude de decision par modele/strategie

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ext_summary["f1"].unstack("strategy").plot.bar(ax=axes[0])
axes[0].set_title("F1 extraction (extracted_requirements vs ground truth)")
axes[0].set_ylabel("F1")
axes[0].set_ylim(0, 1)

decision_acc.unstack("strategy").plot.bar(ax=axes[1])
axes[1].set_title("Exactitude de la decision STAY/OFFLOAD/INFEASIBLE")
axes[1].set_ylabel("accuracy")
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()


## Figure -- Exact Match selon le modele et la strategie de prompting

In [ ]:
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")

MODEL_ORDER = ["gemini-2.5-flash-lite", "gpt-oss-120b", "gpt-oss-20b", "llama-3.3-70b", "mistral-small-3.2-24b"]
STRATEGY_ORDER = ["zero_shot", "one_shot", "few_shot"]
STRATEGY_COLORS = {"zero_shot": "#3569A8", "one_shot": "#28A69A", "few_shot": "#E99A43"}
STRATEGY_LABELS = {"zero_shot": "Zero-shot", "one_shot": "One-shot", "few_shot": "Few-shot"}

FIGURES_DIR = ROOT / "eval" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


def plot_exact_match(col, title, filename):
    pivot = df.groupby(["model_short", "strategy"])[col].mean().unstack("strategy")
    pivot = pivot.reindex(index=MODEL_ORDER, columns=STRATEGY_ORDER)

    fig, ax = plt.subplots(figsize=(8, 5))
    pivot.plot.bar(
        ax=ax, color=[STRATEGY_COLORS[s] for s in STRATEGY_ORDER],
        width=0.75, edgecolor="white", linewidth=0.6,
    )
    ax.set_title(title)
    ax.set_ylabel("Exact Match")
    ax.set_xlabel("")
    ax.set_ylim(0, 1)
    ax.set_xticklabels(pivot.index, rotation=0)
    ax.legend([STRATEGY_LABELS[s] for s in STRATEGY_ORDER], loc="lower right", frameon=True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / filename, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"Sauvegarde : {FIGURES_DIR / filename}")


plot_exact_match("exact_match_ext", "Exact Match selon le modele et la strategie de prompting", "exact_match_ext.png")


In [ ]:
plot_exact_match("exact_match_raw", "Exact Match selon le modele et la strategie de prompting", "exact_match_raw.png")

## Figure 2 -- Effet de la complexite des intentions

Exact Match par modele et par strategie, selon le nombre d'exigences de
l'intent (1 a 4) -- repond a "How does the number of requirements in an
intent affect extraction accuracy?". Deux figures (brut, ext), memes
couleurs/strategies que la Figure 1.

In [ ]:
def plot_exact_match_by_complexity(col, filename):
    pivot = df.groupby(["model_short", "strategy", "n_gt_requirements"])[col].mean().reset_index()

    fig, axes = plt.subplots(1, len(MODEL_ORDER), figsize=(22, 4), sharey=True)
    for ax, model in zip(axes, MODEL_ORDER):
        sub_model = pivot[pivot["model_short"] == model]
        for strategy in STRATEGY_ORDER:
            sub = sub_model[sub_model["strategy"] == strategy].sort_values("n_gt_requirements")
            ax.plot(
                sub["n_gt_requirements"], sub[col],
                marker="o", color=STRATEGY_COLORS[strategy], label=STRATEGY_LABELS[strategy],
                linewidth=2, markersize=5,
            )
        ax.set_title(model)
        ax.set_xlabel("Nombre d'exigences")
        ax.set_xticks([1, 2, 3, 4])
        ax.set_ylim(0, 1)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    axes[0].set_ylabel("Exact Match")
    axes[-1].legend(loc="lower left", frameon=True)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / filename, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"Sauvegarde : {FIGURES_DIR / filename}")


plot_exact_match_by_complexity("exact_match_raw", "exact_match_by_complexity_raw.png")


In [ ]:
plot_exact_match_by_complexity("exact_match_ext", "exact_match_by_complexity_ext.png")


## Figure 3 -- Llama 3.3 70B few-shot (raw) : effet de la taille de l'intention

Deux figures, Llama 3.3 70B few-shot uniquement, sur les metriques RAW (extraction_raw, avant validate_and_normalize) -- choisi comme meilleure config sur l'Exact Match raw (0.962, seul en tete, voir discussion precedente).

In [ ]:
llama_fewshot = (df["model_short"] == "llama-3.3-70b") & (df["strategy"] == "few_shot")
llama_df = df[llama_fewshot]

pivot = llama_df.groupby("n_gt_requirements")["exact_match_raw"].mean().sort_index()

EM_COLOR = "#3569A8"  # scientific blue -- distinct from the Recall/Explicitness figure's orange

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(pivot.index, pivot.values, marker="o", color=EM_COLOR, linewidth=2.5, markersize=8)
for x, y in zip(pivot.index, pivot.values):
    ax.annotate(
        f"{y * 100:.2f}%", (x, y), textcoords="offset points", xytext=(0, 12),
        ha="center", fontsize=11, color=EM_COLOR, fontweight="bold",
    )
ax.set_xlabel("Number of requirements", fontsize=14, fontweight="bold")
ax.set_ylabel("Exact Match", fontsize=14, fontweight="bold")
ax.set_xticks([1, 2, 3, 4])
ax.set_ylim(0.90, 1.02)
ax.set_yticks([0.90, 0.95, 1.00])
ax.tick_params(axis="both", labelsize=12)
for label in ax.get_xticklabels() + ax.get_yticklabels():
    label.set_fontweight("bold")
ax.grid(axis="y", color="#d5d5d5", linewidth=0.7)
ax.grid(axis="x", visible=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "exact_match_by_complexity_llama_fewshot_raw.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"Sauvegarde : {FIGURES_DIR / 'exact_match_by_complexity_llama_fewshot_raw.png'}")


In [ ]:
agg = llama_df.groupby("n_gt_requirements")[["tp_raw", "fp_raw", "fn_raw"]].sum()
agg["precision"] = agg["tp_raw"] / (agg["tp_raw"] + agg["fp_raw"])
agg["recall"] = agg["tp_raw"] / (agg["tp_raw"] + agg["fn_raw"])
agg["f1"] = 2 * agg["precision"] * agg["recall"] / (agg["precision"] + agg["recall"])

METRIC_COLORS = {"precision": "#3569A8", "recall": "#28A69A", "f1": "#E99A43"}
METRIC_LABELS = {"precision": "Precision", "recall": "Recall", "f1": "F1"}

fig, ax = plt.subplots(figsize=(6, 5))
for metric in ["precision", "recall", "f1"]:
    ax.plot(
        agg.index, agg[metric], marker="o", color=METRIC_COLORS[metric],
        label=METRIC_LABELS[metric], linewidth=2.5, markersize=7,
    )
ax.set_xlabel("Number of requirements", fontsize=14, fontweight="bold")
ax.set_ylabel("Score", fontsize=14, fontweight="bold")
ax.set_xticks([1, 2, 3, 4])
ax.set_ylim(0.90, 1.02)
ax.set_yticks([0.90, 0.95, 1.00])
ax.tick_params(axis="both", labelsize=12)
for label in ax.get_xticklabels() + ax.get_yticklabels():
    label.set_fontweight("bold")
legend = ax.legend(loc="lower left", frameon=True, fontsize=12)
for text in legend.get_texts():
    text.set_fontweight("bold")
ax.grid(axis="y", color="#d5d5d5", linewidth=0.7)
ax.grid(axis="x", visible=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "all_metrics_by_complexity_llama_fewshot_raw.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"Sauvegarde : {FIGURES_DIR / 'all_metrics_by_complexity_llama_fewshot_raw.png'}")


## Figure 4 -- Llama 3.3 70B few-shot (raw) : effet du niveau d'explicitation

Rappel (recall) uniquement par niveau L1-L4 -- Precision/F1 ne se decomposent pas proprement par niveau avec notre methodologie de matching actuelle (un faux positif ne correspond a aucune mention de verite terrain, donc n'a pas de niveau attribuable). Sur extraction_raw, pas extracted_requirements (ext).

In [ ]:
llama_mentions = mentions_df[
    (mentions_df["model_short"] == "llama-3.3-70b") & (mentions_df["strategy"] == "few_shot")
]
recall_llama_by_level = llama_mentions.groupby("level")["matched_raw"].mean().sort_index()
level_labels = [f"L{lvl}" for lvl in recall_llama_by_level.index]

RECALL_COLOR = "#E99A43"  # soft orange -- distinct from the Exact Match/Intent size figure's blue

fig, ax = plt.subplots(figsize=(6, 5))
bars = ax.bar(
    level_labels, recall_llama_by_level.values,
    color=RECALL_COLOR, edgecolor="white", linewidth=0.6,
)
for rect, y in zip(bars, recall_llama_by_level.values):
    ax.annotate(
        f"{y * 100:.2f}%", (rect.get_x() + rect.get_width() / 2, y), textcoords="offset points",
        xytext=(0, 8), ha="center", fontsize=11, color="#8a5a1e", fontweight="bold",
    )
ax.set_xlabel("Explicitness level (L1 Explicit ... L4 Oblique)", fontsize=14, fontweight="bold")
ax.set_ylabel("Recall", fontsize=14, fontweight="bold")
ax.set_ylim(0.85, 1.03)
ax.set_yticks([0.85, 0.90, 0.95, 1.00])
ax.tick_params(axis="both", labelsize=12)
for label in ax.get_xticklabels() + ax.get_yticklabels():
    label.set_fontweight("bold")
ax.grid(axis="y", color="#d5d5d5", linewidth=0.7)
ax.grid(axis="x", visible=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "recall_by_level_llama_fewshot_raw.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"Sauvegarde : {FIGURES_DIR / 'recall_by_level_llama_fewshot_raw.png'}")


## Figure 5 -- Temps d'execution et tokens selon la taille de l'intention : notre approche vs baselines

Llama 3.3 70B few-shot partout, trois approches comparees : notre pipeline complet (non ablation, eval/results/), Direct LLM et Chain-of-Thought (eval/baselines/results/). Les baselines font un seul appel LLM (elapsed_s/token_usage uniques) -- notre approche en fait deux (extraction + explication), sommes deja calculees dans perf_df ci-dessus.

In [ ]:
import json as _json

LLAMA_FEWSHOT_FILE = RESULTS_DIR / "meta-llama-llama-3.3-70b-instruct__few_shot__20261103T220001Z.jsonl"
BASELINE_DIR = ROOT / "eval" / "baselines" / "results"

def load_ours_perf(filename):
    rows = []
    with open(filename) as f:
        for line in f:
            rec = _json.loads(line)
            if "ground_truth" not in rec or rec.get("total_elapsed_s") is None:
                continue
            ext_tok = rec.get("extraction_token_usage") or {}
            exp_tok = rec.get("explanation_token_usage") or {}
            rows.append({
                "n_gt_requirements": len(rec["ground_truth"]["requirements"]),
                "total_elapsed_s": rec["total_elapsed_s"],
                "total_tokens": ext_tok.get("total_tokens", 0) + exp_tok.get("total_tokens", 0),
            })
    return pd.DataFrame(rows)

def load_baseline_perf(filename):
    rows = []
    with open(BASELINE_DIR / filename) as f:
        for line in f:
            rec = _json.loads(line)
            if "ground_truth" not in rec or rec.get("elapsed_s") is None:
                continue
            tok = rec.get("token_usage") or {}
            rows.append({
                "n_gt_requirements": len(rec["ground_truth"]["requirements"]),
                "elapsed_s": rec["elapsed_s"],
                "total_tokens": tok.get("total_tokens", 0),
            })
    return pd.DataFrame(rows)

perf_df = load_ours_perf(LLAMA_FEWSHOT_FILE)
direct_perf_df = load_baseline_perf("direct__meta-llama-llama-3.3-70b-instruct__few_shot__20261104T234058Z.jsonl")
cot_perf_df = load_baseline_perf("cot__meta-llama-llama-3.3-70b-instruct__few_shot__20261104T234148Z.jsonl")

APPROACH_COLORS = {"ours": "#3569A8", "direct": "#E99A43", "cot": "#28A69A"}
APPROACH_LABELS = {"ours": "Our approach", "direct": "Direct LLM", "cot": "Chain-of-Thought"}

time_series = {
    "ours": perf_df.groupby("n_gt_requirements")["total_elapsed_s"].mean().sort_index(),
    "direct": direct_perf_df.groupby("n_gt_requirements")["elapsed_s"].mean().sort_index(),
    "cot": cot_perf_df.groupby("n_gt_requirements")["elapsed_s"].mean().sort_index(),
}
tokens_series = {
    "ours": perf_df.groupby("n_gt_requirements")["total_tokens"].mean().sort_index(),
    "direct": direct_perf_df.groupby("n_gt_requirements")["total_tokens"].mean().sort_index(),
    "cot": cot_perf_df.groupby("n_gt_requirements")["total_tokens"].mean().sort_index(),
}
pd.DataFrame(time_series).round(2)


In [ ]:
APPROACH_OFFSETS = {"ours": 12, "direct": -20, "cot": 12}
APPROACH_VA = {"ours": "bottom", "direct": "top", "cot": "bottom"}

fig, ax = plt.subplots(figsize=(7, 5.5))
for approach in ["ours", "direct", "cot"]:
    s = time_series[approach]
    ax.plot(
        s.index, s.values, marker="o", color=APPROACH_COLORS[approach],
        label=APPROACH_LABELS[approach], linewidth=2.5, markersize=8,
    )
    for x, y in zip(s.index, s.values):
        ax.annotate(
            f"{y:.1f}s", (x, y), textcoords="offset points", xytext=(0, APPROACH_OFFSETS[approach]),
            ha="center", va=APPROACH_VA[approach], fontsize=10, color=APPROACH_COLORS[approach], fontweight="bold",
        )
ax.set_xlabel("Number of requirements", fontsize=14, fontweight="bold")
ax.set_ylabel("Execution time (s)", fontsize=14, fontweight="bold")
ax.set_xticks([1, 2, 3, 4])
ax.set_ylim(-1, max(s.max() for s in time_series.values()) * 1.3)
for label in ax.get_xticklabels() + ax.get_yticklabels():
    label.set_fontweight("bold")
ax.tick_params(axis="both", labelsize=12)
legend = ax.legend(loc="upper left", frameon=True, fontsize=12)
for text in legend.get_texts():
    text.set_fontweight("bold")
ax.grid(axis="y", color="#d5d5d5", linewidth=0.7)
ax.grid(axis="x", visible=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "execution_time_by_complexity_comparison.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"Sauvegarde : {FIGURES_DIR / 'execution_time_by_complexity_comparison.png'}")


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5.5))
for approach in ["ours", "direct", "cot"]:
    s = tokens_series[approach]
    ax.plot(
        s.index, s.values, marker="o", color=APPROACH_COLORS[approach],
        label=APPROACH_LABELS[approach], linewidth=2.5, markersize=8,
    )
    for x, y in zip(s.index, s.values):
        ax.annotate(
            f"{y:,.0f}", (x, y), textcoords="offset points", xytext=(0, 10),
            ha="center", fontsize=10, color=APPROACH_COLORS[approach], fontweight="bold",
        )
ax.set_xlabel("Number of requirements", fontsize=14, fontweight="bold")
ax.set_ylabel("Total tokens", fontsize=14, fontweight="bold")
ax.set_xticks([1, 2, 3, 4])
ax.set_ylim(0, max(s.max() for s in tokens_series.values()) * 1.3)
for label in ax.get_xticklabels() + ax.get_yticklabels():
    label.set_fontweight("bold")
ax.tick_params(axis="both", labelsize=12)
legend = ax.legend(loc="upper left", frameon=True, fontsize=12)
for text in legend.get_texts():
    text.set_fontweight("bold")
ax.grid(axis="y", color="#d5d5d5", linewidth=0.7)
ax.grid(axis="x", visible=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "total_tokens_by_complexity_comparison.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"Sauvegarde : {FIGURES_DIR / 'total_tokens_by_complexity_comparison.png'}")


## Inspection d'erreurs -- quelques cas de decision incorrecte

Pour verification manuelle, pas un metrique -- un echantillon de scenarios
ou la decision predite ne correspond pas a la verite terrain.


In [ ]:
wrong = df[~df["decision_correct"]]
print(f"{len(wrong)} decisions incorrectes sur {len(df)} ({len(wrong)/len(df):.1%})")
display(wrong.sample(min(10, len(wrong)), random_state=0)[
    ["model_short", "strategy", "scenario_id", "gt_decision", "pred_decision"]
])


## Scenarios encore en echec (exclus du scoring ci-dessus)

Les 8 scenarios `gpt-oss-120b` qui echouent encore malgre le relevement de
`max_tokens` a 8192 -- generation incontrolee, pas un probleme de calibrage.


In [ ]:
still_failing = []
for name in RESULT_FILES:
    with open(RESULTS_DIR / name) as f:
        for line in f:
            r = json.loads(line)
            if "error" in r:
                still_failing.append({"file": name, "scenario_id": r["scenario_id"]})

pd.DataFrame(still_failing)
